### AI 실험용 파일

- 이미지 데이터 불러오기
- meal / program / other 라벨 확인
- 이미지 전처리
- 사전학습 모델 테스트
- 이미지 분류 결과 확인
- 게시글 자동 생성 테스트
- 모델 저장

In [ ]:
import random
import shutil
from pathlib import Path

# Ultralytics classify 학습용 폴더 구조를 만들어서
# train/val에 "비어있는 클래스"가 생기지 않게 합니다.
# (현재 data/walking 폴더처럼 이미지가 0개면 자동 제외)

SRC = Path("data")
DST = Path("data_cls")  # 생성될 데이터셋 루트
TRAIN_RATIO = 0.8
SEED = 42

random.seed(SEED)

all_dirs = [p for p in SRC.iterdir() if p.is_dir()]
if not all_dirs:
    # 이름 바꾸는 
    raise FileNotFoundError(f"No class folders under: {SRC.resolve()}")

def list_images(folder: Path):
    images = []
    for ext in ["*.png", "*.jpg", "*.jpeg", "*.webp"]:
        images.extend(folder.glob(ext))
    return sorted(images)

# 이미지가 1개 이상인 클래스만 사용 (0개면 스킵)
classes = []
skipped = []
for d in all_dirs:
    imgs = list_images(d)
    if len(imgs) == 0:
        skipped.append(d.name)
        continue
    classes.append((d, imgs))

if skipped:
    print("[WARN] empty classes skipped:", skipped)

if len(classes) < 2:
    raise ValueError("Need at least 2 non-empty classes to train.")

# 기존 결과가 있으면 삭제 후 재생성
if DST.exists():
    shutil.rmtree(DST)

for split in ["train", "val"]:
    for (cdir, _imgs) in classes:
        (DST / split / cdir.name).mkdir(parents=True, exist_ok=True)

for (cdir, images) in classes:
    # 클래스별로 분할해서 train/val에 반드시 들어가도록 보장
    random.shuffle(images)

    if len(images) == 1:
        # 데이터가 1장뿐이면 학습은 가능하게(중복) but 품질은 낮음
        train_imgs = images
        val_imgs = images
    else:
        n_val = max(1, int(round(len(images) * (1 - TRAIN_RATIO))))
        n_train = max(1, len(images) - n_val)
        if n_train + n_val > len(images):
            n_val = len(images) - n_train

        train_imgs = images[:n_train]
        val_imgs = images[n_train:]
        if not val_imgs:
            val_imgs = [train_imgs[-1]]

    for p in train_imgs:
        shutil.copy2(p, DST / "train" / cdir.name / p.name)
    for p in val_imgs:
        shutil.copy2(p, DST / "val" / cdir.name / p.name)

print("Prepared dataset:")
print("- root:", DST.resolve())
for split in ["train", "val"]:
    total = sum(1 for _ in (DST / split).rglob("*.*"))
    cls_counts = {
        cdir.name: sum(1 for _ in (DST / split / cdir.name).rglob("*.*"))
        for (cdir, _imgs) in classes
    }
    print(f"- {split}: {total} images", cls_counts)

In [1]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

# 위 셀에서 만든 data_cls/train, data_cls/val 구조를 사용
model.train(
    data="data_cls",
    epochs=30,
    imgsz=224,
    batch=8,
    name="program_action_cls"
)

Ultralytics 8.4.56  Python-3.14.3 torch-2.12.0+cpu CPU (AMD Ryzen 5 8640HS w/ Radeon 760M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=program_action_cls, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

RuntimeError: Dataset 'data' error  [WinError 3]     : 'C:\\Gabia2026\\FinalProject\\HiddencoreFinal\\ai\\ai-server\\datasets\\data\\train'